In [22]:
import os
import glob
import re
import joblib
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

import sklearn

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.3.2


In [28]:
GENERATED_DIR = "/home/muhamed/repo/CTDDG/Results/generated_103"

RF_MODEL_PATH = "/home/muhamed/repo/CTDDG/code/random_forest_model.joblib"

OUTPUT_DIR = "/home/muhamed/repo/CTDDG/Results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "toxicity_distribution_103.csv"
)

DETAILED_OUTPUT = os.path.join(
    OUTPUT_DIR,
    "toxicity_predictions_detailed_103.csv"
)

print("Generated directory:", GENERATED_DIR)
print("RF model:", RF_MODEL_PATH)
print("Summary output:", SUMMARY_OUTPUT)

Generated directory: /home/muhamed/repo/CTDDG/Results/generated_103
RF model: /home/muhamed/repo/CTDDG/code/random_forest_model.joblib
Summary output: /home/muhamed/repo/CTDDG/Results/toxicity_distribution_103.csv


In [29]:
rf = joblib.load(RF_MODEL_PATH)

print("Model loaded successfully!")
print("Model type:", type(rf))
print("Number of features:", rf.n_features_in_)
print("Classes:", rf.classes_)
print("Number of trees:", rf.n_estimators)

Model loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Number of features: 2048
Classes: [0. 1.]
Number of trees: 100


In [30]:
def smiles_to_morgan(smiles, radius=2, n_bits=2048):
    """
    Convert a SMILES string into a 2048-bit Morgan fingerprint.
    Returns None if the SMILES is invalid.
    """

    if pd.isna(smiles):
        return None

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    fingerprint = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits
    )

    array = np.zeros((n_bits,), dtype=np.uint8)

    DataStructs.ConvertToNumpyArray(
        fingerprint,
        array
    )

    return array

In [32]:
csv_files = sorted(
    glob.glob(
        os.path.join(
            GENERATED_DIR,
            "protein*",
            "*_all_samples.csv"
        )
    )
)

print("Number of CSV files found:", len(csv_files))

for path in csv_files[:10]:
    print(path)

Number of CSV files found: 103
/home/muhamed/repo/CTDDG/Results/generated_103/protein1/protein1_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein10/protein10_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein100/protein100_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein101/protein101_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein102/protein102_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein103/protein103_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein11/protein11_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein12/protein12_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein13/protein13_all_samples.csv
/home/muhamed/repo/CTDDG/Results/generated_103/protein14/protein14_all_samples.csv


In [33]:
summary_results = []
detailed_results = []

for csv_path in csv_files:

    # -----------------------------------------
    # Protein name
    # -----------------------------------------

    protein_name = os.path.basename(
        os.path.dirname(csv_path)
    )

    print(f"Processing {protein_name}...")

    # -----------------------------------------
    # Read CSV
    # -----------------------------------------

    df = pd.read_csv(csv_path)

    if "SMILES" not in df.columns:
        print(f"  WARNING: SMILES column missing")
        continue

    # -----------------------------------------
    # Generate fingerprints
    # -----------------------------------------

    fingerprints = []
    valid_indices = []
    invalid_indices = []

    for idx, smiles in enumerate(df["SMILES"]):

        fp = smiles_to_morgan(smiles)

        if fp is None:
            invalid_indices.append(idx)
        else:
            fingerprints.append(fp)
            valid_indices.append(idx)

    # -----------------------------------------
    # RF prediction
    # -----------------------------------------

    if len(fingerprints) > 0:

        X = np.array(fingerprints)

        predictions = rf.predict(X)

        toxic_count = int(
            np.sum(predictions == 1)
        )

        non_toxic_count = int(
            np.sum(predictions == 0)
        )

    else:

        predictions = np.array([])

        toxic_count = 0
        non_toxic_count = 0

    # -----------------------------------------
    # Counts
    # -----------------------------------------

    total_candidates = len(df)
    valid_candidates = len(fingerprints)
    invalid_smiles = len(invalid_indices)

    # -----------------------------------------
    # Percentages
    # -----------------------------------------

    if valid_candidates > 0:

        toxic_percentage = (
            toxic_count /
            valid_candidates *
            100
        )

        non_toxic_percentage = (
            non_toxic_count /
            valid_candidates *
            100
        )

    else:

        toxic_percentage = 0
        non_toxic_percentage = 0

    # -----------------------------------------
    # Summary row
    # -----------------------------------------

    summary_results.append({
        "Protein": protein_name,
        "Total Candidates": total_candidates,
        "Valid Candidates": valid_candidates,
        "Invalid SMILES": invalid_smiles,
        "Toxic Candidates": toxic_count,
        "Non-Toxic Candidates": non_toxic_count,
        "Toxic %": round(toxic_percentage, 2),
        "Non-Toxic %": round(non_toxic_percentage, 2)
    })

    # -----------------------------------------
    # Detailed results
    # -----------------------------------------

    for local_idx, original_idx in enumerate(valid_indices):

        detailed_results.append({
            "Protein": protein_name,
            "Run_ID": df.iloc[original_idx]["Run_ID"],
            "SMILES": df.iloc[original_idx]["SMILES"],
            "Toxicity_Label": int(predictions[local_idx])
        })

    print(
        f"  Total={total_candidates}, "
        f"Valid={valid_candidates}, "
        f"Toxic={toxic_count}, "
        f"Non-toxic={non_toxic_count}"
    )

Processing protein1...
  Total=100, Valid=100, Toxic=30, Non-toxic=70
Processing protein10...
  Total=100, Valid=100, Toxic=39, Non-toxic=61
Processing protein100...
  Total=100, Valid=100, Toxic=41, Non-toxic=59
Processing protein101...
  Total=100, Valid=100, Toxic=38, Non-toxic=62
Processing protein102...
  Total=100, Valid=100, Toxic=31, Non-toxic=69
Processing protein103...
  Total=100, Valid=100, Toxic=30, Non-toxic=70
Processing protein11...
  Total=100, Valid=100, Toxic=31, Non-toxic=69
Processing protein12...
  Total=100, Valid=100, Toxic=27, Non-toxic=73
Processing protein13...
  Total=100, Valid=100, Toxic=36, Non-toxic=64
Processing protein14...
  Total=100, Valid=100, Toxic=32, Non-toxic=68
Processing protein15...
  Total=100, Valid=100, Toxic=30, Non-toxic=70
Processing protein16...
  Total=100, Valid=100, Toxic=30, Non-toxic=70
Processing protein17...
  Total=100, Valid=100, Toxic=38, Non-toxic=62
Processing protein18...
  Total=100, Valid=100, Toxic=34, Non-toxic=66
Pro

In [34]:
summary_df = pd.DataFrame(summary_results)

# Sort protein1, protein2, ..., protein104 numerically
summary_df["Protein_Number"] = (
    summary_df["Protein"]
    .str.extract(r"(\d+)")
    .astype(int)
)

summary_df = summary_df.sort_values(
    "Protein_Number"
)

summary_df = summary_df.drop(
    columns=["Protein_Number"]
)

summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False
)

print("Saved summary:")
print(SUMMARY_OUTPUT)

print()
print(summary_df)

Saved summary:
/home/muhamed/repo/CTDDG/Results/toxicity_distribution_103.csv

        Protein  Total Candidates  Valid Candidates  Invalid SMILES  \
0      protein1               100               100               0   
15     protein2               100               100               0   
26     protein3               100               100               0   
37     protein4               100               100               0   
48     protein5               100               100               0   
..          ...               ...               ...             ...   
102   protein99               100               100               0   
2    protein100               100               100               0   
3    protein101               100               100               0   
4    protein102               100               100               0   
5    protein103               100               100               0   

     Toxic Candidates  Non-Toxic Candidates  Toxic %  Non-Toxic %  


In [35]:
detailed_df = pd.DataFrame(detailed_results)

detailed_df.to_csv(
    DETAILED_OUTPUT,
    index=False
)

print("Detailed predictions saved to:")
print(DETAILED_OUTPUT)

print()
print(detailed_df.head(20))

Detailed predictions saved to:
/home/muhamed/repo/CTDDG/Results/toxicity_predictions_detailed_103.csv

     Protein  Run_ID                                             SMILES  \
0   protein1       1                   CC1(C)c2ccccc2CC(c2cnccn2)N1CCCO   
1   protein1       1  CCn1c(C2=CCN(S(=O)(=O)c3ccccc3)CC2)nc2ccc(-c3c...   
2   protein1       1                O=C(CO)N1CCC2(CCCCC2(O)c2ccccn2)CC1   
3   protein1       1  O=C(Nc1cnc(Nc2ccc(Cl)cc2S(=O)(=O)N2CCOCC2)c2cc...   
4   protein1       1                 COC(=O)C1=CC(c2cccc(OC)c2OC)C(C)N1   
5   protein1       1    Fc1ccccc1-c1c[nH]c(-c2ccnc3ccc(C(F)(F)F)nc23)n1   
6   protein1       1          Cc1nc2cc(-n3ccnc3CCS(C)(=O)=O)ccn2c1C(C)C   
7   protein1       1                       CCCN1CCCCC1Cc1c[nH]c2ncccc12   
8   protein1       1  CCCNS(=O)(=O)c1ccc2c(c1)C(N(C(=O)CCl)c1ccccc1C...   
9   protein1       1  CC(C)C(C)NC(=O)c1cc([N+](=O)[O-])c(-n2[nH]c3nc...   
10  protein1       1  CCOC1c2ccccc2C(=O)N1c1cc(-c2ccc(C)cc2)n(-c2ccc... 